# 02 — Feature Engineering

**Purpose:** construct leakage-safe lifecycle snapshots from the canonical daily dataset. Future outcomes are targets only; current scoring snapshots contain no `target_*` columns.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
from game_performance.config import SETTINGS
from game_performance.features import build_all_historical_snapshots, categorical_features, model_features
from game_performance.io import load_dataframe, save_artifact

df = load_dataframe(SETTINGS.processed_dir / "clean_game_daily.pkl")


In [3]:
historical_snapshots = build_all_historical_snapshots(df, SETTINGS.horizon_strategy)
summary = {key: len(frame) for key, frame in historical_snapshots.items()}
summary


{(21, 7): 8311,
 (28, 7): 7153,
 (35, 7): 7017,
 (42, 7): 7684,
 (49, 7): 7435,
 (56, 7): 7332}

In [4]:
example_key = next(iter(historical_snapshots))
example = historical_snapshots[example_key]
print("Example configuration:", example_key)
print("Categorical features:", categorical_features(example))
print("Model feature count:", len(model_features(example)))
example.head()


Example configuration: (21, 7)
Categorical features: ['SystemName', 'TournamentName', 'Launch_Month', 'Launch_Weekday', 'Observation_End_Month', 'Observation_End_Weekday', 'Clients_Size_Band', 'Stakes_Size_Band']
Model feature count: 206


,Game_ID,SystemName,TournamentName,First_Observed_Date,Launch_Month,Launch_Weekday,Observation_End_Month,Observation_End_Weekday,Observation_End_DayOfMonth,Observation_End_WeekOfYear,...,log_betcount_week_2_sum,log_betcount_week_3_sum,Clients_Size_Band,Stakes_Size_Band,target_clients,target_stakes,target_clients_week_1,target_clients_week_2,target_stakes_week_1,target_stakes_week_2
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,AmusnetSlots,AmusnetS-100 Bulky Fruits,2026-02-01,February,Sunday,February,Saturday,21,8,...,10.799167,10.564809,Large,Large,479,190018.0,479,795,190018.0,241854.0
1,AmusnetSlots::AmusnetS-100 Cats,AmusnetSlots,AmusnetS-100 Cats,2026-02-01,February,Sunday,February,Saturday,21,8,...,14.346879,13.309318,Very large,Very large,4914,1114534.0,4914,3760,1114534.0,1078553.0
2,AmusnetSlots::AmusnetS-100 Power Hot,AmusnetSlots,AmusnetS-100 Power Hot,2026-02-01,February,Sunday,February,Saturday,21,8,...,14.644580,13.665397,Very large,Very large,8437,6685104.0,8437,6781,6685104.0,1998301.0
3,AmusnetSlots::AmusnetS-2 Dragons,AmusnetSlots,AmusnetS-2 Dragons,2026-02-01,February,Sunday,February,Saturday,21,8,...,11.045000,10.730357,Large,Large,859,115100.6,859,1274,115100.6,177381.4
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,AmusnetSlots,AmusnetS-20 Bulky Fruits,2026-02-01,February,Sunday,February,Saturday,21,8,...,13.481030,12.360565,Very large,Very large,2299,845772.0,2299,1919,845772.0,387345.4


In [5]:
save_artifact(historical_snapshots, SETTINGS.feature_dir / "historical_snapshots.joblib")
print("Saved feature snapshots")


Saved feature snapshots
